<a href="https://colab.research.google.com/github/MacKumachin/Clifford-FBSM-SignalControl/blob/main/Algebraic_Crowding_Hubs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Rigorous Numerical Simulation for Stochastic Boundary Ideals and Relativistic Control
This script reproduces the exact numerical results for Figures 7.1, 7.2, and 7.3.
- Fig 7.1: Stratonovich SDE integration with Ito drift correction
- Fig 7.2: Monte Carlo simulation (10,000 paths) for First Passage Time distributions
- Fig 7.3: Forward-Backward Sweep Method (FBSM) for Stochastic Pontryagin Maximum Principle
"""

import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# Global Plot Style Settings (Academic Journal Standard)
# ==========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'figure.dpi': 300, # PDF出力用の高解像度
    'savefig.bbox': 'tight'
})

# ==========================================
# Fig 7.1: Rigorous Stratonovich SDE Trajectory
# ==========================================
def compute_and_save_fig1():
    print("Computing Fig 7.1: Stratonovich SDE integration...")
    np.random.seed(42)

    dt = 0.005
    n_steps = 5000
    w = np.zeros(n_steps)
    k = np.zeros(n_steps)

    # 初期値（安定領域内部の右上）
    w[0] = 1.8
    k[0] = 2.5

    mu_w, mu_k = 0.0, 0.3
    alpha = 0.5

    for i in range(1, n_steps):
        dW_w = np.sqrt(dt) * np.random.randn()
        dW_k = np.sqrt(dt) * np.random.randn()

        sigma_w = alpha * w[i-1]
        sigma_k = alpha * k[i-1]

        # Ito drift correction for Stratonovich SDE
        ito_drift_w = 0.5 * alpha * sigma_w
        ito_drift_k = 0.5 * alpha * sigma_k

        drift_w = -mu_w * w[i-1]
        drift_k = -mu_k

        w[i] = w[i-1] + (drift_w + ito_drift_w) * dt + sigma_w * dW_w
        k[i] = k[i-1] + (drift_k + ito_drift_k) * dt + sigma_k * dW_k

        # Breakdown condition (crossing Arnold tongue boundary)
        if k[i] < np.abs(w[i]):
            w = w[:i+1]
            k = k[:i+1]
            break

    fig, ax = plt.subplots(figsize=(8, 6))
    w_grid = np.linspace(-3, 3, 500)
    k_bound = np.abs(w_grid)
    ax.fill_between(w_grid, k_bound, 4.0, color='#a8c8f0', alpha=0.6, label='Synchronised Region (Arnold Tongue)')
    ax.plot(w_grid, k_bound, 'k--', linewidth=1.5, alpha=0.7)

    ax.plot(0, 0, 'ko', markersize=10, zorder=5, label='Crowding Hub (OT > 1)')
    ax.plot(w, k, color='#e63946', linewidth=1.5, label='Rigorous Stratonovich Trajectory')
    ax.plot(w[0], k[0], 'go', markersize=8, label='Start')
    ax.plot(w[-1], k[-1], 'rX', markersize=12, zorder=6, label='Noise-Induced Escape')

    ax.set_title("Fig 7.1: Rigorous Stochastic Phase Space & Invisible Wedges")
    ax.set_xlabel(r"Detuning $\Delta\omega$")
    ax.set_ylabel(r"Coupling Strength $\kappa$")
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(-0.2, 3.0)

    # 【修正箇所】完全に空白となる「右下」に凡例を配置
    ax.legend(loc='lower right', framealpha=0.9, edgecolor='gray')

    ax.grid(True, linestyle=':', alpha=0.7)

    filename = 'Fig_7_1_PhaseSpace.pdf'
    plt.savefig(filename)
    print(f"  -> Saved as {filename}")
    plt.close()

# ==========================================
# Fig 7.2: Monte Carlo First Passage Time Distribution
# ==========================================
def compute_and_save_fig2():
    print("Computing Fig 7.2: Monte Carlo FPT integration (10,000 paths)...")
    np.random.seed(42)

    N_paths = 10000
    dt = 0.05
    T_max = 500.0
    n_steps = int(T_max / dt)

    phi_normal = np.zeros(N_paths)
    phi_hub = np.zeros(N_paths)
    fpt_normal = np.full(N_paths, np.nan)
    fpt_hub = np.full(N_paths, np.nan)

    active_normal = np.ones(N_paths, dtype=bool)
    active_hub = np.ones(N_paths, dtype=bool)

    w_normal, k_normal = 1.0, 0.6
    w_hub_init, k_hub_init = 1.0, 0.6
    alpha = 0.01
    sigma = 0.4

    for step in range(n_steps):
        t = step * dt
        dW = np.sqrt(dt) * np.random.randn(N_paths)

        if np.any(active_normal):
            drift_n = w_normal - k_normal * np.sin(phi_normal[active_normal])
            phi_normal[active_normal] += drift_n * dt + sigma * dW[active_normal]
            crossed_n = active_normal & (np.abs(phi_normal) > np.pi/2)
            fpt_normal[crossed_n] = t
            active_normal[crossed_n] = False

        if np.any(active_hub):
            w_hub = w_hub_init * np.exp(-alpha * t)
            k_hub = k_hub_init * np.exp(-alpha * t)
            drift_h = w_hub - k_hub * np.sin(phi_hub[active_hub])
            phi_hub[active_hub] += drift_h * dt + sigma * dW[active_hub]
            crossed_h = active_hub & (np.abs(phi_hub) > np.pi/2)
            fpt_hub[crossed_h] = t
            active_hub[crossed_h] = False

        if not np.any(active_normal) and not np.any(active_hub):
            break

    fpt_normal = fpt_normal[~np.isnan(fpt_normal)]
    fpt_hub = fpt_hub[~np.isnan(fpt_hub)]

    fig, ax = plt.subplots(figsize=(8, 6))
    bins = np.logspace(0, np.log10(T_max), 50)

    ax.hist(fpt_normal, bins=bins, density=True, alpha=0.5, color='#457b9d', label='Normal Boundary ($\mathrm{OT}=1$)')
    ax.hist(fpt_hub, bins=bins, density=True, alpha=0.5, color='#e63946', label='Crowding Hub ($\mathrm{OT}>1$)')

    ax.set_title("Fig 7.2: Rigorous First Passage Time Distributions (Monte Carlo)")
    ax.set_xlabel(r"Time until synchronisation loss $\tau$")
    ax.set_ylabel("Probability Density")
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend(framealpha=0.9)
    ax.grid(True, which="both", linestyle=':', alpha=0.6)

    filename = 'Fig_7_2_FPT_Dist.pdf'
    plt.savefig(filename)
    print(f"  -> Saved as {filename}")
    plt.close()

# ==========================================
# Fig 7.3: Forward-Backward Sweep Method for PPG
# ==========================================
def compute_and_save_fig3():
    print("Computing Fig 7.3: Forward-Backward Sweep Method (FBSM)...")
    np.random.seed(111)

    T = 30.0
    dt = 0.02
    N = int(T / dt)
    time = np.linspace(0, T, N)

    w = 1.0
    sigma = 0.5
    gamma = 0.5

    dW = np.sqrt(dt) * np.random.randn(N)
    phi_free = np.zeros(N)
    k_free = 0.2
    for i in range(1, N):
        phi_free[i] = phi_free[i-1] + (w - k_free * np.sin(phi_free[i-1])) * dt + sigma * dW[i]

    kappa_opt = np.ones(N) * 0.2
    phi = np.zeros(N)
    p = np.zeros(N)

    max_iter = 30
    tolerance = 1e-4
    learning_rate = 0.1

    for it in range(max_iter):
        kappa_old = np.copy(kappa_opt)

        # Forward Sweep
        phi[0] = 0.0
        for i in range(1, N):
            phi[i] = phi[i-1] + (w - kappa_opt[i-1] * np.sin(phi[i-1])) * dt + sigma * dW[i]

        # Backward Sweep
        p[-1] = 0.0
        for i in range(N-2, -1, -1):
            dp_dt = p[i+1] * kappa_opt[i+1] * np.cos(phi[i+1]) + 2.0 * phi[i+1]
            p[i] = p[i+1] - dp_dt * dt

        # Update Control
        kappa_new = - p * np.sin(phi) / (2.0 * gamma)
        kappa_new = np.clip(kappa_new, 0.0, 4.0)
        kappa_opt = (1 - learning_rate) * kappa_old + learning_rate * kappa_new

        if np.max(np.abs(kappa_opt - kappa_old)) < tolerance:
            print(f"  -> FBSM Converged at iteration {it+1}")
            break

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

    ax1.plot(time, phi_free, color='#9ca3af', linestyle='--', linewidth=1.5, label='Uncontrolled (Phase Slip)')
    ax1.plot(time, phi, color='#1d3557', linewidth=2, label='Controlled (FBSM Rigorous)')

    ax1.axhline(np.pi/2, color='#e63946', linestyle=':', linewidth=2, label=r'Algebraic Boundary $\mathcal{V}(\mathcal{I}_{bd})$')
    ax1.axhline(-np.pi/2, color='#e63946', linestyle=':', linewidth=2)

    ax1.set_title("Fig 7.3: Strict Stochastic Optimal Control via FBSM")
    ax1.set_ylabel(r"Phase Difference $\Delta\varphi$")
    ax1.legend(loc='upper left', framealpha=0.9)
    ax1.grid(True, linestyle=':', alpha=0.7)

    ax2.plot(time, kappa_opt, color='#e63946', linewidth=2, label=r'Optimal Coupling $\kappa^*(t)$')
    ax2.fill_between(time, 0, kappa_opt, color='#e63946', alpha=0.2)
    ax2.set_xlabel("Time $t$")
    ax2.set_ylabel(r"Control $\kappa$")
    ax2.legend(loc='upper right', framealpha=0.9)
    ax2.grid(True, linestyle=':', alpha=0.7)

    filename = 'Fig_7_3_OptimalControl.pdf'
    plt.savefig(filename)
    print(f"  -> Saved as {filename}")
    plt.close()

# ==========================================
# Main Execution Block
# ==========================================
if __name__ == "__main__":
    print("Starting rigorous numerical simulations...\n")
    compute_and_save_fig1()
    compute_and_save_fig2()
    compute_and_save_fig3()
    print("\nAll simulations completed successfully. PDF files have been generated.")

<>:149: SyntaxWarning: invalid escape sequence '\m'
<>:150: SyntaxWarning: invalid escape sequence '\m'
<>:149: SyntaxWarning: invalid escape sequence '\m'
<>:150: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipython-input-1239575532.py:149: SyntaxWarning: invalid escape sequence '\m'
  ax.hist(fpt_normal, bins=bins, density=True, alpha=0.5, color='#457b9d', label='Normal Boundary ($\mathrm{OT}=1$)')
/tmp/ipython-input-1239575532.py:150: SyntaxWarning: invalid escape sequence '\m'
  ax.hist(fpt_hub, bins=bins, density=True, alpha=0.5, color='#e63946', label='Crowding Hub ($\mathrm{OT}>1$)')


Starting rigorous numerical simulations...

Computing Fig 7.1: Stratonovich SDE integration...
  -> Saved as Fig_7_1_PhaseSpace.pdf
Computing Fig 7.2: Monte Carlo FPT integration (10,000 paths)...
  -> Saved as Fig_7_2_FPT_Dist.pdf
Computing Fig 7.3: Forward-Backward Sweep Method (FBSM)...
  -> Saved as Fig_7_3_OptimalControl.pdf

All simulations completed successfully. PDF files have been generated.
